# SuttaPlayer Piper1 VITS Training Console (v4)
This notebook serves as your interactive control panel for training the non-rhotic Australian male Sutta voice model on Google Colab's Free Tier T4 GPU. This pipeline utilizes an offline Deno orchestrator, compiles the Cython Monotonic Alignment Search (MAS) C-extension natively, manages an isolated Python 3.11.9 environment (aligned with modern PyTorch 2.x and PyTorch Lightning 2.x), handles background daemons under TMUX, and runs active validation probes with 100% pre-phonemized targets (fully stripped of old carrier signal pads and calibrated against Audacity).

## Step 1: Mount Google Drive
Mount your Google Drive to expose your training corpus to the Colab container.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Provision Deno Runtime
Download and install the Deno security-isolated environment to run the training manager.

In [ ]:
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'
!deno --version

## Step 3: Run Environment Initialization
Execute the Deno training manager script. This will automatically:\n1. Download and compile **Micromamba**.\n2. Provision an isolated **Python 3.11.9** environment.\n3. Clone `OHF-Voice/piper1-gpl` and install dependencies (`torch==2.3.1`, `onnx==1.15.0`, `lightning==2.3.3`).\n4. **Compile the Cython Monotonic Alignment Search (MAS) C-Extension** natively inside the virtual environment for 10x faster training loops.\n5. Prepare local fast-SSD cache paths under `/content/piper_cache` to shield Google Drive from I/O bottlenecks.

In [ ]:
# Copy the training manager from your Drive base to Colab space and run the init pipeline
!cp /content/drive/MyDrive/piper_training/sutta-training-manager-v4.ts /content/sutta-training-manager-v4.ts
!deno run --allow-all /content/sutta-training-manager-v4.ts --init

## Step 4: Start TensorBoard (Lightweight progress monitoring)
Launch TensorBoard to play synthesized probe WAV files from validation epochs and track global scalar loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/piper_cache/lightning_logs

## Step 5: Start Training inside TMUX Daemon
Starts the training thread inside a secure TMUX terminal. It integrates `train_sutta_voice.py` as an active callback.

In [ ]:
!deno run --allow-all /content/sutta-training-manager-v4.ts --train

## Step 6: Start Real-Time Synchronizer & Keep-Alive Heartbeat
Runs the real-time background sync daemon. This will:\n1. Sync checkpoints (`.ckpt`) back to Google Drive every 10 seconds (rsync).\n2. Parse the training execution metrics (`metrics.csv`) from version logs.\n3. Issue keep-alive pings to prevent the Google Colab session from timing out or disconnecting.

In [ ]:
!deno run --allow-all /content/sutta-training-manager-v4.ts --monitor